# 🏛️ NeuralOps Quickstart: 1-Line Surgical Attention Replacement & Micro-Healing

This tutorial demonstrates how to replace standard Quadratic Multi-Head Attention in a pretrained Transformer (e.g. GPT-2) with subquadratic linear attention (`gated_delta_net`), calibrate representation alignment with closed-form Ridge regression, and run layerwise subspace micro-healing without catastrophic forgetting.

In [ ]:
!pip install neuralops torch transformers
import torch
import neuralops
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"NeuralOps Version: {neuralops.__version__}")
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Active Device: {device}")

## 1. Load Pretrained Host Model

In [ ]:
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()
print(f"Loaded {model_name} with {sum(p.numel() for p in model.parameters()):,} parameters.")

## 2. 1-Line Attention Surgery
Replace quadratic self-attention with Triton-fused `gated_delta_net`.

In [ ]:
import copy
teacher_model = copy.deepcopy(model)

# Perform zero-friction transplant
neuralops.replace_attention(model, target_module="gated_delta_net")
print("Transplant complete! Attention layers replaced with GatedDeltaNet.")

## 3. Layerwise Subspace Micro-Healing
Align representations using cached activations and closed-form Ridge warm-start.

In [ ]:
# Generate calibration tokens
prompt = "NeuralOps provides subquadratic linear attention mechanisms."
calib_tokens = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

# Heal student model directly on representations
stats = neuralops.heal_replaced_model(
    teacher_model=teacher_model,
    student_model=model,
    calibration_inputs=calib_tokens,
    steps=15,
    lr=2e-3,
)
print(f"Initial Loss: {stats.get('initial_mse', 0.0):.4f} -> Final Loss: {stats.get('final_mse', 0.0):.4f}")

## 4. Save Bit-Exact Surgical Checkpoint

In [ ]:
import tempfile
save_dir = tempfile.mkdtemp(prefix="neuralops_checkpoint_")
saved_path = neuralops.save_surgical_checkpoint(model, save_dir)
print(f"Model and surgical manifest saved to: {saved_path}")